# 介紹
本程式使用 pyautogui 來完成一個 Blitz 的遊戲機器人。

In [5]:
import pyautogui as pyg
pyg.sleep(2)
basex=1389
basey=492
gap=40

In [6]:
for row in range(8):
    for col in range(8):
        x = basex + col * gap
        y = basey + row * gap
        color = pyg.pixel(x, y)
        print(row, col, x, y, color)
        

0 0 1389 492 (25, 26, 27)
0 1 1429 492 (25, 26, 27)
0 2 1469 492 (25, 26, 27)
0 3 1509 492 (25, 26, 27)
0 4 1549 492 (42, 43, 44)
0 5 1589 492 (25, 26, 27)
0 6 1629 492 (25, 26, 27)
0 7 1669 492 (25, 26, 27)
1 0 1389 532 (32, 33, 34)
1 1 1429 532 (32, 33, 34)
1 2 1469 532 (32, 33, 34)
1 3 1509 532 (32, 33, 34)
1 4 1549 532 (125, 125, 125)
1 5 1589 532 (18, 19, 20)
1 6 1629 532 (25, 26, 27)
1 7 1669 532 (25, 26, 27)
2 0 1389 572 (25, 26, 27)
2 1 1429 572 (25, 26, 27)
2 2 1469 572 (25, 26, 27)
2 3 1509 572 (25, 26, 27)
2 4 1549 572 (25, 26, 27)
2 5 1589 572 (25, 26, 27)
2 6 1629 572 (25, 26, 27)
2 7 1669 572 (25, 26, 27)
3 0 1389 612 (25, 26, 27)
3 1 1429 612 (25, 26, 27)
3 2 1469 612 (25, 26, 27)
3 3 1509 612 (25, 26, 27)
3 4 1549 612 (25, 26, 27)
3 5 1589 612 (25, 26, 27)
3 6 1629 612 (25, 26, 27)
3 7 1669 612 (25, 26, 27)
4 0 1389 652 (25, 26, 27)
4 1 1429 652 (25, 26, 27)
4 2 1469 652 (25, 26, 27)
4 3 1509 652 (25, 26, 27)
4 4 1549 652 (25, 26, 27)
4 5 1589 652 (25, 26, 27)
4 6 1629 

In [7]:
def get_gem_position(r, c):
    """回傳第 r 列、第 c 欄寶石中心的螢幕座標 (x, y)；r、c 範圍為 0 到 7。"""
    if not (0 <= r < 8 and 0 <= c < 8):
        raise ValueError("r 和 c 必須是 0 到 7 的整數")

    x = basex + c * gap
    y = basey + r * gap
    return (x, y)

In [4]:
def get_gem_color(r, c, im):
    """回傳截圖 im 中第 r 列、第 c 欄寶石中心點的顏色。"""
    x, y = get_gem_position(r, c)
    return im.getpixel((x, y))

In [ ]:
def can_move_left(r, c, im):
    """判斷第 r 列、第 c 欄的寶石左移一格後是否會形成三連線。"""
    if not (isinstance(r, int) and isinstance(c, int) and 0 <= r < 8 and 0 < c < 8):
        return False

    # 從截圖取得目前 8x8 棋盤的顏色。
    board = [[get_gem_color(row, col, im) for col in range(8)] for row in range(8)]

    def find_matches(gems):
        matches = set()
        # 檢查所有水平的連續三顆。
        for row in range(8):
            for col in range(6):
                color = gems[row][col]
                if color == gems[row][col + 1] == gems[row][col + 2]:
                    matches.add(("h", row, col, color))
        # 檢查所有垂直的連續三顆。
        for row in range(6):
            for col in range(8):
                color = gems[row][col]
                if color == gems[row + 1][col] == gems[row + 2][col]:
                    matches.add(("v", row, col, color))
        return matches

    matches_before = find_matches(board)
    board[r][c], board[r][c - 1] = board[r][c - 1], board[r][c]
    matches_after = find_matches(board)

    # 只有交換後新形成三連線才算可以移動。
    return bool(matches_after - matches_before)

In [ ]:
def _can_swap_make_match(r1, c1, r2, c2, im):
    """判斷交換兩顆寶石後，是否會新形成三連線。"""
    if not (0 <= r1 < 8 and 0 <= c1 < 8 and 0 <= r2 < 8 and 0 <= c2 < 8):
        return False

    board = [[get_gem_color(row, col, im) for col in range(8)] for row in range(8)]

    def find_matches(gems):
        matches = set()
        for row in range(8):
            for col in range(6):
                color = gems[row][col]
                if color == gems[row][col + 1] == gems[row][col + 2]:
                    matches.add(("h", row, col, color))
        for row in range(6):
            for col in range(8):
                color = gems[row][col]
                if color == gems[row + 1][col] == gems[row + 2][col]:
                    matches.add(("v", row, col, color))
        return matches

    matches_before = find_matches(board)
    board[r1][c1], board[r2][c2] = board[r2][c2], board[r1][c1]
    return bool(find_matches(board) - matches_before)


def can_move_right(r, c, im):
    return c < 7 and can_move_left(r, c + 1, im)


def can_move_up(r, c, im):
    return _can_swap_make_match(r, c, r - 1, c, im)


def can_move_down(r, c, im):
    return _can_swap_make_match(r, c, r + 1, c, im)


import time
start_time = time.time()
im = pyg.screenshot()

while True:
    if time.time() - start_time >= 60:
        break

    moved = False
    for r in range(8):
        if time.time() - start_time >= 60:
            break
        for c in range(8):
            if time.time() - start_time >= 60:
                break

            directions = [
                (can_move_left, 0, -1),
                (can_move_right, 0, 1),
                (can_move_up, -1, 0),
                (can_move_down, 1, 0),
            ]
            for can_move, dr, dc in directions:
                if can_move(r, c, im):
                    start_pos = get_gem_position(r, c)
                    end_pos = get_gem_position(r + dr, c + dc)
                    pyg.moveTo(*start_pos)
                    pyg.dragTo(*end_pos, duration=0.1, button="left")
                    pyg.sleep(0.15)
                    im = pyg.screenshot()
                    moved = True
                    break
            if moved:
                break

    if time.time() - start_time >= 60:
        break
    if not moved:
        pyg.sleep(0.1)
        im = pyg.screenshot()